1. Data Pipeline: Face Extraction to Drive

In [ ]:
import os
import cv2
import random
import numpy as np
from mtcnn import MTCNN
from PIL import Image
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive to save the extracted dataset persistently
drive.mount('/content/drive')

# Define source paths where the extracted Celeb-DF dataset is located
source_real = "/content/dataset/Celeb-real"
source_fake = "/content/dataset/Celeb-synthesis"

# Define base destination path for the new balanced dataset
base_save_dir = "/content/drive/MyDrive/Deepfake_Project/Balanced_Dataset_V2"
save_real_dir = os.path.join(base_save_dir, "Real")
save_fake_dir = os.path.join(base_save_dir, "Fake")

# Create destination directories if they don't exist
os.makedirs(save_real_dir, exist_ok=True)
os.makedirs(save_fake_dir, exist_ok=True)

# Initialize the MTCNN face detector
detector = MTCNN()

# Define quotas based on the 10% few-shot requirement (balanced 50/50)
TARGET_QUOTA = 311
FRAMES_PER_VIDEO = 32

def process_balanced_set(source_folder, save_folder, quota, label):
    videos = [v for v in os.listdir(source_folder) if v.endswith('.mp4')]
    random.seed(42)
    random.shuffle(videos)

    # 1. Check how many videos are ALREADY processed in the Drive
    existing_folders = [d for d in os.listdir(save_folder) if os.path.isdir(os.path.join(save_folder, d))]
    success_count = len(existing_folders)

    pbar = tqdm(total=quota, desc=f"Processing {label}")
    pbar.update(success_count) # Start the progress bar from where we left off

    for video_name in videos:
        if success_count >= quota:
            break

        video_id = video_name.split('.')[0]
        video_dir = os.path.join(save_folder, video_id)

        # 2. THE SKIP LOGIC: If folder exists, skip to the next video
        if os.path.exists(video_dir):
            continue

        video_path = os.path.join(source_folder, video_name)
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames < FRAMES_PER_VIDEO:
            cap.release()
            continue

        indices = np.linspace(0, total_frames - 1, FRAMES_PER_VIDEO, dtype=int)
        faces_captured = []

        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret: break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            try:
                results = detector.detect_faces(frame_rgb)
                if results:
                    x, y, w, h = results[0]['box']
                    face = frame_rgb[max(0, y-20):y+h+20, max(0, x-20):x+w+20]
                    if face.shape[0] > 0 and face.shape[1] > 0:
                        faces_captured.append(Image.fromarray(face))
            except: continue

        cap.release()

        if len(faces_captured) == FRAMES_PER_VIDEO:
            os.makedirs(video_dir, exist_ok=True)
            for i, img in enumerate(faces_captured):
                img.save(os.path.join(video_dir, f"frame_{i:02d}.jpg"))
            success_count += 1
            pbar.update(1)

    pbar.close()
    return success_count
# Execute the pipeline for both real and fake datasets
print("--- Starting Balanced Data Extraction ---")
real_count = process_balanced_set(source_real, save_real_dir, TARGET_QUOTA, "Real")
fake_count = process_balanced_set(source_fake, save_fake_dir, TARGET_QUOTA, "Fake")

print(f"\nExtraction Complete!")
print(f"Total Real Videos: {real_count}")
print(f"Total Fake Videos: {fake_count}")

In [ ]:
# 0.6 Data Sanity Check
import glob

real_dir = '/content/Balanced_Dataset_V2/Real'
fake_dir = '/content/Balanced_Dataset_V2/Fake'

real_images = glob.glob(os.path.join(real_dir, '**', '*.*'), recursive=True)
real_images = [p for p in real_images if p.lower().endswith(('.png', '.jpg', '.jpeg'))]

fake_images = glob.glob(os.path.join(fake_dir, '**', '*.*'), recursive=True)
fake_images = [p for p in fake_images if p.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f" REAL: {len(real_images)} | FAKE: {len(fake_images)} | Total: {len(real_images)+len(fake_images)}")